# 01 — Anatomie de la cible : quand TAXITIME n'est pas un taxi

Ce notebook produit les figures 1 à 3 du papier depuis les caches du pipeline
(`data/dep_features_cache_v15.parquet`, `data/val_preds_v15.parquet`).
Prérequis : avoir lancé `reproduce.py` une fois (les caches sont créés au premier run).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("..") / "data"
dep = pd.read_parquet(DATA / "dep_features_cache_v15.parquet")
print(f"{len(dep):,} departs, colonnes: {dep.shape[1]}")

## Figure 1 — Deux définitions d'off-block : biais aéroport vs Network Manager

`delta_mvt_aobt3_s` est le taxi vu par NM (`MVT − AOBT_3`), `TAXITIME` le taxi
vu par l'aéroport. Leur écart médian par aéroport montre que les deux systèmes
ne mesurent pas la même chose.

In [ ]:
d = dep[(dep["is_corrupt"] == 0) & dep["delta_mvt_aobt3_s"].notna()].copy()
d["ecart"] = d["delta_mvt_aobt3_s"] - d["TAXITIME_SEC_mvt"]
stats = d.groupby("ADEP_mvt", observed=True)["ecart"].agg(["median", "std", "size"]).sort_values("median")
print(stats.round(0))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(stats.index.astype(str), stats["median"], yerr=stats["std"] / np.sqrt(stats["size"]),
       color="#3B82F6")
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("Ecart median NM − aeroport (s)")
ax.set_title("Biais entre l'off-block NM (AOBT_3) et l'off-block aeroport, lignes saines")
plt.tight_layout(); plt.show()

## Figure 2 — La substitution silencieuse par l'horaire

4,8 % des départs ont `BLOCK = SCHED` à 10 s près ; leur `TAXITIME` vaut alors
`MVT − SCHED` (retard + taxi réel), pas un taxi.

In [ ]:
taux = dep.groupby("ADEP_mvt", observed=True)["is_corrupt"].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(taux.index.astype(str), 100 * taux.values, color="#10B981")
ax.axhline(100 * dep["is_corrupt"].mean(), color="black", ls="--", lw=0.8,
           label=f"moyenne {100 * dep['is_corrupt'].mean():.1f} %")
ax.set_ylabel("Part des departs a BLOCK = SCHED (%)")
ax.set_title("Taux de substitution de l'off-block par l'horaire, 2025")
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Verification de l'identite TAXITIME = MVT − SCHED sur ces lignes
c = dep[dep["is_corrupt"] == 1]
ecart = (c["TAXITIME_SEC_mvt"] - c["delta_mvt_sched_s"]).abs()
print(f"|TAXITIME − (MVT − SCHED)| sur les lignes substituees : "
      f"mediane {ecart.median():.0f} s, p95 {ecart.quantile(0.95):.0f} s")

## Figure 3 — Le RMSE est décidé par une poignée de lignes

Décomposition du MSE de validation (janvier + juillet 2025, prédictions du
pipeline final) par régime, et courbe de concentration.

In [ ]:
v = pd.read_parquet(DATA / "val_preds_v15.parquet")
v["se"] = (v["pred"] - v["y"]) ** 2
N = len(v); mse = v["se"].sum() / N
v["regime"] = np.select(
    [v["is_corrupt"] == 1, (v["y"] - 86400).between(30, 7200), v["y"] > 3600],
    ["block = sched", "bug +1 jour", "queue > 1 h"], "normal")
part = v.groupby("regime")["se"].sum() / v["se"].sum()
n_par = v.groupby("regime").size()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
a1.barh(part.index, 100 * part.values, color="#3B82F6")
for i, (p, n) in enumerate(zip(part.values, n_par.values)):
    a1.text(100 * p + 1, i, f"{n:,} lignes", va="center", fontsize=9)
a1.set_xlabel("Part du MSE (%)"); a1.set_title(f"MSE par regime (RMSE {mse**0.5:.0f} s)")

s = v["se"].sort_values(ascending=False).cumsum() / v["se"].sum()
ks = np.arange(1, 2001)
a2.plot(ks, 100 * s.iloc[:2000].values, color="#10B981")
a2.set_xscale("log"); a2.set_xlabel("k pires lignes (log)")
a2.set_ylabel("Part cumulee du MSE (%)")
a2.set_title("Concentration : 10 lignes ≈ 30 % du MSE")
a2.axvline(10, color="black", ls=":", lw=0.8)
plt.tight_layout(); plt.show()

Lecture : plus de la moitié du MSE vient du régime normal (le plancher de
données), un quart de quelques queues sans signature, et les régimes
« substitués » sont largement traités par le mélange.